In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

load_dotenv()

In [ ]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str


llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    max_tokens=300, 
)

In [ ]:
def create_outline(state: BlogState) -> BlogState:
    title = state['title']
    prompt = f"Generate an outline for blog on title {title}. Do not use markdown"

    outline = llm.invoke(prompt).text
    state['outline'] = outline

    return state


def create_blog(state: BlogState) -> BlogState:
    title = state['title']
    outline = state['outline']
    prompt = f"Write a blog on title {title} with following outline \n{outline} \n\nDo not use markdown"

    content = llm.invoke(prompt).text
    state['content'] = content

    return state

In [ ]:
graph = StateGraph(BlogState)

graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()

In [ ]:
initial_state = {'title': 'History of Moon'}

final_state = workflow.invoke(initial_state)

In [ ]:
print(final_state['content'])

In [ ]:
# see graph, following code only work in notebook
from IPython.display import Image
Image(workflow.get_graph().draw_mermaid_png())